In [ ]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [ ]:
# Данный блок в области 1 выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
#
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
#
# pip install

In [2]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, OneHotEncoder, PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error
import pickle


### Область работы 2 (выполнение лучшей модели)

In [ ]:
# Данный блок(и) в области 2 выполняется преподавателем
#
# В области находится одна, единственная, итоговая модель машинного обучения с однозначными, 
# зафиксированными параметрами
#
# В данной области категорически запрещается искать, выбирать, улучшать, оптимизировать, 
# тюниговать и т.д. модель машинного обучения

In [3]:
# Путь к тренировочному набору
path_train = 'train.csv' # содержит только имя файла, без имен папок
# Путь к тестовому набору
path_test  = 'test.csv' # содержит только имя файла, без имен папок

df = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)
df_test

,size,color,fancy_color_dominant_color,fancy_color_secondary_color,fancy_color_overtone,fancy_color_intensity,clarity,cut,symmetry,polish,...,table_percent,meas_length,meas_width,meas_depth,girdle_min,girdle_max,culet_condition,fluor_color,fluor_intensity,eye_clean
0,0.35,G,NaN,NaN,NaN,NaN,VVS2,Excellent,Excellent,Excellent,...,56.0,4.50,4.53,2.83,NaN,NaN,NaN,NaN,NaN,NaN
1,0.21,E,NaN,NaN,NaN,NaN,SI1,Excellent,Excellent,Excellent,...,59.0,3.82,3.84,2.34,M,STK,NaN,NaN,NaN,Yes
2,0.33,F,NaN,NaN,NaN,NaN,SI2,Excellent,Excellent,Excellent,...,57.0,4.41,4.44,2.78,M,STK,NaN,NaN,NaN,Yes
3,0.32,F,NaN,NaN,NaN,NaN,SI1,Excellent,Excellent,Excellent,...,57.0,4.38,4.41,2.73,M,STK,NaN,NaN,Faint,Borderline
4,0.80,F,NaN,NaN,NaN,NaN,VS2,Excellent,Excellent,Excellent,...,55.0,5.88,5.92,3.71,M,STK,NaN,NaN,Faint,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26329,0.30,H,NaN,NaN,NaN,NaN,VS1,Excellent,Very Good,Excellent,...,57.0,4.25,4.29,2.66,M,STK,NaN,NaN,Faint,Yes
26330,0.30,J,NaN,NaN,NaN,NaN,SI1,Very Good,Very Good,Excellent,...,58.0,4.29,4.32,2.69,TN,STK,NaN,Blue,Strong,NaN
26331,0.55,D,NaN,NaN,NaN,NaN,SI2,Excellent,Excellent,Excellent,...,57.0,5.28,5.31,3.22,TN,STK,NaN,Blue,Medium,NaN
26332,0.50,H,NaN,NaN,NaN,NaN,VVS2,Excellent,Excellent,Excellent,...,57.0,5.03,5.07,3.18,M,STK,NaN,Blue,Medium,NaN


In [ ]:
df = df.drop_duplicates()

In [ ]:
# Блок обучения модели

In [ ]:
X = df.drop(columns=['total_sales_price'])
y = df['total_sales_price']

drop_cols = [
    'fancy_color_dominant_color', 'fancy_color_secondary_color', 
    'fancy_color_overtone', 'fancy_color_intensity',
    'culet_condition', 'fluor_color'
]
df = df.drop(columns=drop_cols, errors='ignore')

num_features = [
    'size', 'depth_percent', 'table_percent', 
    'meas_length', 'meas_width', 'meas_depth'
]
cat_features = [
    'color', 'clarity', 'cut', 'symmetry', 'polish', 
    'girdle_min', 'girdle_max', 'fluor_intensity', 'eye_clean'
]

num_transformer = Pipeline(steps=[
    ('imputer', IterativeImputer(max_iter=20)),
    ('poly', PolynomialFeatures(2, include_bias=False)),
    ('scaler', RobustScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

CT = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
]).set_output(transform='pandas')

pipeline = Pipeline(steps=[
    ('preproc', CT),
    ('estimator', SVR())
])

model_target = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=PowerTransformer(method='yeo-johnson')
)

model_target.fit(X, y)

In [ ]:
# Блок предсказания с использованием тестового набора

In [ ]:

y_predict = model_target.predict(df_test)